In [2]:
from pathlib import Path

import pandas as pd

# Lokasi file
file_fitur = Path(
    r"D:\00 Kuliah\TA\Experiment\Model\5 ML\Ekstraksi_fitur_squeezenet.csv"
)
file_target = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan_dengan_jumlah_sel.csv"
)
file_hasil = file_fitur.with_name("dataset_fitur_jumlah_sel.csv")

kolom_nama_gambar_fitur = "nama_file"
kolom_nama_gambar_target = "Nama_Gambar"
kolom_jumlah_sel = "Jumlah Sel (sel/mL)"

# Baca sebagai teks supaya nama gambar dan angka bertitik tidak berubah format
df_fitur = pd.read_csv(file_fitur, encoding="utf-8-sig", dtype=str)
df_target = pd.read_csv(file_target, encoding="utf-8-sig", dtype=str)

# Periksa kolom yang dibutuhkan
for kolom in (kolom_nama_gambar_fitur,):
    if kolom not in df_fitur.columns:
        raise KeyError(f"Kolom '{kolom}' tidak ditemukan di file fitur.")

for kolom in (kolom_nama_gambar_target, kolom_jumlah_sel):
    if kolom not in df_target.columns:
        raise KeyError(f"Kolom '{kolom}' tidak ditemukan di file jumlah sel.")

kolom_fitur = [
    kolom for kolom in df_fitur.columns
    if kolom.startswith("fitur_")
]
if not kolom_fitur:
    raise ValueError("Tidak ditemukan kolom fitur dengan awalan 'fitur_'.")

# Buat key nama file yang dinormalisasi untuk proses pencocokan
df_fitur["_key_gambar"] = (
    df_fitur[kolom_nama_gambar_fitur].astype("string").str.strip().str.lower()
)
df_target["_key_gambar"] = (
    df_target[kolom_nama_gambar_target].astype("string").str.strip().str.lower()
)

if df_fitur["_key_gambar"].isna().any() or df_fitur["_key_gambar"].eq("").any():
    raise ValueError("Ada baris fitur yang tidak memiliki nama file gambar.")
if df_target["_key_gambar"].isna().any() or df_target["_key_gambar"].eq("").any():
    raise ValueError("Ada baris target yang tidak memiliki Nama_Gambar.")

# Pastikan setiap gambar hanya muncul satu kali pada masing-masing file
if df_fitur["_key_gambar"].duplicated().any():
    raise ValueError("Ada nama gambar duplikat di file fitur.")
if df_target["_key_gambar"].duplicated().any():
    raise ValueError("Ada Nama_Gambar duplikat di file jumlah sel.")

# Angka target di CSV memakai titik sebagai pemisah ribuan.
# Hapus titik agar target dibaca sebagai angka, misalnya "121.847.700" -> 121847700.
jumlah_sel_teks = (
    df_target[kolom_jumlah_sel]
    .astype("string")
    .str.replace(".", "", regex=False)
)
df_target[kolom_jumlah_sel] = pd.to_numeric(jumlah_sel_teks, errors="raise")

# Gabungkan fitur dan target berdasarkan nama gambar
df_hasil = df_fitur[
    [kolom_nama_gambar_fitur, "_key_gambar", *kolom_fitur]
].merge(
    df_target[["_key_gambar", kolom_jumlah_sel]],
    on="_key_gambar",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

# Hentikan proses jika ada gambar yang tidak berpasangan
tidak_cocok = df_hasil["_merge"] != "both"
if tidak_cocok.any():
    contoh = df_hasil.loc[tidak_cocok, "_key_gambar"].head(10).tolist()
    raise ValueError(
        f"Ada {tidak_cocok.sum()} gambar yang tidak memiliki pasangan. "
        f"Contoh nama file: {contoh}"
    )

# Simpan hanya nama gambar, fitur, dan target jumlah sel
df_hasil = df_hasil[
    [kolom_nama_gambar_fitur, *kolom_fitur, kolom_jumlah_sel]
]

df_hasil.to_csv(file_hasil, index=False, encoding="utf-8-sig")

print(f"Jumlah baris hasil: {len(df_hasil)}")
print(f"Jumlah fitur per gambar: {len(kolom_fitur)}")
print(f"File hasil disimpan di: {file_hasil}")
display(df_hasil.head())

Jumlah baris hasil: 101
Jumlah fitur per gambar: 169000
File hasil disimpan di: D:\00 Kuliah\TA\Experiment\Model\5 ML\dataset_fitur_jumlah_sel.csv


,nama_file,fitur_000001,fitur_000002,fitur_000003,fitur_000004,fitur_000005,fitur_000006,fitur_000007,fitur_000008,fitur_000009,...,fitur_168992,fitur_168993,fitur_168994,fitur_168995,fitur_168996,fitur_168997,fitur_168998,fitur_168999,fitur_169000,Jumlah Sel (sel/mL)
0,ITS-POME_2025-10-20_17-00-05.jpg,0.459522008895874,3.4756295680999756,2.9164371490478516,3.6024606227874756,1.545616626739502,0.48496943712234497,0.0,1.0310485363006592,1.554656982421875,...,3.6512808799743652,2.9504635334014893,2.5335030555725098,1.8178112506866455,1.1565101146697998,1.6417136192321777,7.059842109680176,11.19687271118164,12.788568496704102,93619100
1,ITS-POME_2025-10-20_17-05-05.jpg,2.0886247158050537,4.819874286651611,4.163033485412598,2.867534875869751,0.16652125120162964,0.0,0.0,0.0,0.0909871757030487,...,2.99238657951355,2.39982271194458,1.8362233638763428,1.3188602924346924,0.8876508474349976,2.4613335132598877,8.358203887939453,14.419487953186035,15.067169189453125,121847700
2,ITS-POME_2025-10-20_17-10-05.jpg,3.0237491130828857,4.9583258628845215,2.394026279449463,1.5320053100585938,2.1295721530914307,0.48731622099876404,0.3641568124294281,0.40465906262397766,1.2346220016479492,...,2.032564163208008,1.7306503057479858,1.990926742553711,1.9547393321990967,1.7893511056900024,2.1576504707336426,5.827088832855225,10.84779167175293,11.689451217651367,134297800
3,ITS-POME_2025-10-20_17-15-05.jpg,2.141031265258789,4.108057975769043,3.847172260284424,3.523310661315918,3.226346015930176,0.8980532288551331,0.37297579646110535,0.0,0.0,...,2.627549171447754,1.6230294704437256,1.4621591567993164,1.81044602394104,1.9049619436264038,2.9759609699249268,7.854665756225586,11.867570877075195,11.836137771606445,163217200
4,ITS-POME_2025-10-20_17-20-05.jpg,2.7910778522491455,3.9174227714538574,2.3617608547210693,1.211604356765747,1.347961187362671,1.8168015480041504,1.0643961429595947,0.0,0.2542582154273987,...,2.749901294708252,2.713192939758301,2.5356035232543945,2.243682384490967,1.6421818733215332,2.4809255599975586,7.074822425842285,12.221786499023438,11.930097579956055,72832300
